# Task 2: Lightweight QLoRA Instruction Fine-Tuning

This notebook demonstrates a practical instruction fine-tuning workflow for a Senior Machine Learning Engineer assessment. The goal is not to chase benchmarks. The goal is to show reproducible, explainable, free-tier-compatible GenAI engineering.

We fine-tune a small instruct model with QLoRA, evaluate fixed prompts before and after tuning, save LoRA adapters locally, and optionally upload the adapter to Hugging Face Hub.

## 1. Introduction

QLoRA combines two practical ideas:

- **4-bit quantization**: load the base model with lower-precision weights to reduce GPU memory.
- **LoRA adapters**: train a small number of adapter parameters while keeping the base model frozen.

This makes instruction tuning possible on constrained GPUs such as free Colab runtimes.

## 2. Environment Setup

Use a GPU runtime for training. In Colab, install dependencies if needed. The helper module uses lazy imports, so it can be inspected on CPU, but model loading/training expects GPU-compatible `bitsandbytes`.

In [4]:
# Colab only: uncomment if dependencies are missing.
# !pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece

from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "task2_genai" else Path.cwd()
# TASK2_SRC = PROJECT_ROOT / "task2_genai" / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from task2_genai.src.finetuning_workflow import (
    compare_before_after,
    create_instruction_dataset,
    ensure_task2_directories,
    export_default_training_config,
    load_base_model_and_tokenizer,
    load_instruction_dataset_for_training,
    load_json,
    load_model_with_adapter,
    load_training_config,
    prepare_model_for_lora,
    push_adapter_to_hub,
    resolve_config_paths,
    save_evaluation_results,
    save_sample_prompts,
    train_qlora_adapters,
    generate_response,
)

ensure_task2_directories()
config = resolve_config_paths(export_default_training_config())
config

{'base_model': 'Qwen/Qwen2.5-3B-Instruct',
 'adapter_output_dir': 'C:\\Users\\poshi\\Documents\\My\\Interview-quiz\\cdazzdev-mla\\task2_genai\\outputs\\qwen2_5_3b_task2_lora',
 'dataset_path': 'C:\\Users\\poshi\\Documents\\My\\Interview-quiz\\cdazzdev-mla\\task2_genai\\data\\instruction_dataset.jsonl',
 'evaluation_path': 'C:\\Users\\poshi\\Documents\\My\\Interview-quiz\\cdazzdev-mla\\task2_genai\\evaluation\\before_vs_after_results.json',
 'sample_prompts_path': 'C:\\Users\\poshi\\Documents\\My\\Interview-quiz\\cdazzdev-mla\\task2_genai\\data\\sample_prompts.json',
 'max_seq_length': 768,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 4,
 'learning_rate': 0.0002,
 'warmup_ratio': 0.03,
 'logging_steps': 5,
 'save_strategy': 'epoch',
 'lora_r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'target_modules': ['q_proj',
  'k_proj',
  'v_proj',
  'o_proj',
  'gate_proj',
  'up_proj',
  'down_proj'],
 'load_in_4bit': True,
 'bnb_4bit_quant_type'

## 3. Dataset Preparation

The dataset uses a simple instruction/response format. A small high-quality dataset is appropriate here because the goal is to teach response style and format, not broad new knowledge.

Each row has:

- `instruction`: user request
- `response`: desired assistant answer

During preprocessing, rows are converted to the model's chat template with `tokenizer.apply_chat_template()`.

In [5]:
dataset_rows = create_instruction_dataset(Path(config["dataset_path"]))
sample_prompts = save_sample_prompts(Path(config["sample_prompts_path"]))

print(f"Training examples: {len(dataset_rows)}")
print(f"Evaluation prompts: {len(sample_prompts)}")
dataset_rows[:3]

Training examples: 12
Evaluation prompts: 4


[{'instruction': 'Explain QLoRA to a junior machine learning engineer.',
  'response': 'QLoRA fine-tunes a quantized language model using small LoRA adapter weights. The base model is loaded in 4-bit precision to reduce GPU memory, while only the adapter matrices are trained. This makes instruction tuning practical on limited hardware.'},
 {'instruction': 'Summarize why structured outputs improve AI workflows.',
  'response': 'Structured outputs make model responses easier to validate, log, test, and pass between components. Instead of parsing free-form text, downstream code can rely on predictable fields and handle missing or invalid values safely.'},
 {'instruction': 'Give three production risks in an LLM application.',
  'response': 'Three production risks are malformed model outputs, hidden prompt or data drift, and external API failures. Good systems mitigate these with validation, observability, fallbacks, and monitoring.'}]

## 4. Baseline Model Evaluation

Before training, generate baseline outputs on fixed prompts. Keeping prompts fixed makes the before/after comparison fair and reproducible.

In [6]:
# This cell downloads the base model and should be run on a GPU runtime.
base_model, tokenizer = load_base_model_and_tokenizer(config)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

c:\Users\poshi\Documents\My\Interview-quiz\cdazzdev-mla\.venv\Lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\poshi\Documents\My\Interview-quiz\cdazzdev-mla\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
baseline_responses = []
for prompt in sample_prompts:
    response = generate_response(
        base_model,
        tokenizer,
        prompt,
        max_new_tokens=config["max_new_tokens"],
    )
    baseline_responses.append(response)
    print("PROMPT:", prompt)
    print("BASELINE:", response)
    print("-" * 80)

c:\Users\poshi\Documents\My\Interview-quiz\cdazzdev-mla\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\poshi\Documents\My\Interview-quiz\cdazzdev-mla\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


KeyboardInterrupt: 

## 5. QLoRA Fine-Tuning

The model is prepared for k-bit training, then LoRA adapters are attached. Only adapter parameters are trained.

Important tradeoffs:

- Lower memory than full fine-tuning
- Smaller artifacts because adapters are saved separately
- Less flexible than updating all model weights
- Can overfit quickly on tiny datasets, so keep epochs conservative

In [ ]:
train_dataset = load_instruction_dataset_for_training(
    config["dataset_path"],
    tokenizer,
)
train_dataset[0]["text"][:500]

In [ ]:
qlora_model = prepare_model_for_lora(base_model, config)
qlora_model.print_trainable_parameters()

In [ ]:
trainer = train_qlora_adapters(
    qlora_model,
    tokenizer,
    train_dataset,
    config,
)

print("Saved adapter to:", config["adapter_output_dir"])

## 6. Post-Training Evaluation

Now generate responses from the adapter model using the same prompts. The comparison is qualitative because this dataset is intentionally small.

In [ ]:
# If you restarted the notebook after training, use:
# tuned_model, tokenizer = load_model_with_adapter(config)

tuned_model = qlora_model
fine_tuned_responses = []
for prompt in sample_prompts:
    response = generate_response(
        tuned_model,
        tokenizer,
        prompt,
        max_new_tokens=config["max_new_tokens"],
    )
    fine_tuned_responses.append(response)
    print("PROMPT:", prompt)
    print("FINE-TUNED:", response)
    print("-" * 80)

## 7. Sample Generations

The structured comparison below is saved under `task2_genai/evaluation/` for reproducibility.

In [ ]:
comparison_rows = compare_before_after(
    sample_prompts,
    baseline_responses,
    fine_tuned_responses,
)

evaluation_path = save_evaluation_results(
    comparison_rows,
    Path(config["evaluation_path"]),
)
print("Saved evaluation to:", evaluation_path)
comparison_rows

## 8. Saving Adapters

LoRA adapters are saved separately from the base model. This is practical because the adapter is much smaller than the full model and can be loaded on top of the original base model later.

In [ ]:
adapter_dir = Path(config["adapter_output_dir"])
print("Adapter directory:", adapter_dir)
print("Files:", list(adapter_dir.glob("*")) if adapter_dir.exists() else "Adapter not found yet")

## 9. Optional Hugging Face Upload

Upload only if you want to publish or back up the adapter. Authenticate first with `huggingface_hub.notebook_login()`.

In [ ]:
# HF upload.
from huggingface_hub import notebook_login
notebook_login()
push_adapter_to_hub(tuned_model, tokenizer, repo_id="poshitha/task2-qwen-qlora-adapter", private=True)

## 10. Engineering Reflection

### Why QLoRA was chosen

QLoRA is suitable for free-tier GPUs because 4-bit quantization reduces memory usage while LoRA limits the number of trainable parameters.

### Why PEFT is practical

PEFT trains adapters instead of all model weights. This reduces GPU memory, training time, storage cost, and upload size.

### Free-tier GPU limitations

Free GPUs have limited VRAM, session timeouts, variable availability, and slower training. The config uses batch size 1, gradient accumulation, one epoch, and short sequence length to stay practical.

### Tradeoffs vs full fine-tuning

Full fine-tuning can adapt more deeply but requires much more compute and storage. LoRA is cheaper and easier to manage, but it may be less expressive for large behavior changes.

### Why small high-quality datasets can work

For style and format adaptation, clear examples often matter more than volume. A tiny dataset will not teach broad knowledge, but it can teach the expected response pattern.

### Common mistakes

- Training without baseline outputs
- Changing evaluation prompts after tuning
- Using too many epochs on a tiny dataset
- Forgetting to save the tokenizer with adapters
- Treating qualitative improvement as benchmark proof
- Running 4-bit training on CPU-only environments

### Hugging Face integration

The adapter can be pushed to the Hub after authentication. Users load the base model and then attach the adapter with PEFT.

### Engineering Summary

This workflow implements a reproducible QLoRA fine-tuning pipeline using:
- fixed training configuration
- a small curated instruction dataset
- baseline model evaluation
- parameter-efficient adapter training
- structured before/after comparison

The design prioritizes practical fine-tuning on limited hardware and reproducible engineering workflows over large-scale experimentation.